# ***Tenno: Data Privacy Act***

## **Falcon-7B QLoRA Fine-Tuning Notebook**


## **Objective**

---


This notebook performs parameter-efficient fine-tuning (PEFT) on Falcon-7B using QLoRA (4-bit quantization + LoRA adapters). Training samples are constructed from the frozen baseline prompt set by retrieving relevant legal context (FAISS) and using the expected outputs as supervised targets. After fine-tuning, the model is evaluated on the same prompts under the same retrieval setup and outputs are saved for comparison in the main notebook.

# **Section 1: Enviroment Setup**

---

## **1.1 Install Required Packages**

---

In [1]:
!pip uninstall -y torch torchvision torchaudio
!pip cache purge
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q faiss-cpu
!pip install -q -U "bitsandbytes>=0.46.1"

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 115.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 117.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 86.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 57.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 107.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
    

Installs the PEFT fine-tuning stack (LoRA), quantization (bitsandbytes), dataset utilities, retrieval tooling (FAISS), and sentence embeddings.

## **1.2 Mount Google Drive**

---

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


Mounts Google Drive so we can load FAISS artifacts, prompts, and store trained adapters + outputs.

## **1.3 Persistent Hugging Face Cache (Drive)**

---

In [3]:
import os

PROJECT_DIR = "/content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project"
HF_CACHE_DIR = f"{PROJECT_DIR}/Models/hf_cache"
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["HUGGINGFACE_HUB_CACHE"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

print("✅ Using persistent HF cache:", HF_CACHE_DIR)

✅ Using persistent HF cache: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Models/hf_cache


Ensures model downloads persist across sessions.

## **1.4 Define Paths + Load Prompts and FAISS Artifacts**

---

In [4]:
import json
import faiss

PROMPTS_FILE = f"{PROJECT_DIR}/Prompts/baseline_prompts_v1.json"
FAISS_INDEX_PATH = f"{PROJECT_DIR}/Chunks/faiss_index.bin"
FAISS_META_PATH  = f"{PROJECT_DIR}/Chunks/faiss_metadata.json"

RESULTS_DIR = f"{PROJECT_DIR}/Results"
os.makedirs(RESULTS_DIR, exist_ok=True)

ADAPTER_DIR = f"{PROJECT_DIR}/Models/falcon_finetune_adapter"
os.makedirs(ADAPTER_DIR, exist_ok=True)

with open(PROMPTS_FILE, "r", encoding="utf-8") as f:
    prompts = json.load(f)

index = faiss.read_index(FAISS_INDEX_PATH)
with open(FAISS_META_PATH, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("✅ Prompts:", len(prompts))
print("✅ FAISS vectors:", index.ntotal)
print("✅ Meta rows:", len(meta))
print("✅ Adapter output dir:", ADAPTER_DIR)

✅ Prompts: 10
✅ FAISS vectors: 198
✅ Meta rows: 198
✅ Adapter output dir: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Models/falcon_finetune_adapter


Loads frozen prompts and retrieval artifacts.

### **1.4.1 Artifact Safety and Versioning**

In [5]:
import os, shutil, time

def safe_copy_file(src, dst):
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f"✅ Copied: {src} -> {dst}")
    else:
        print(f"⚠️ Not found (skip): {src}")

def safe_move_dir(src, dst):
    if os.path.exists(src):
        if os.path.exists(dst):
            shutil.rmtree(dst)
            print(f"🧹 Removed existing: {dst}")
        shutil.move(src, dst)
        print(f"✅ Moved: {src} -> {dst}")
    else:
        print(f"⚠️ Not found (skip): {src}")

# Canonical/original filenames
PHASE1_JSON_CANON = f"{RESULTS_DIR}/finetuned_outputs_falcon.json"
PHASE1_ADAPTER_CANON = f"{PROJECT_DIR}/Models/falcon_finetune_adapter"

# Frozen copy names
PHASE1_JSON_ORIGINAL = f"{RESULTS_DIR}/finetuned_outputs_falcon_phase1_original.json"
PHASE1_ADAPTER_ORIGINAL = f"{PROJECT_DIR}/Models/falcon_finetune_adapter_phase1_original"

# Rerun names
PHASE1_JSON_RERUN = f"{RESULTS_DIR}/finetuned_outputs_falcon_phase1_rerun.json"
PHASE1_ADAPTER_RERUN = f"{PROJECT_DIR}/Models/falcon_finetune_adapter_phase1_rerun"

print("✅ Version paths prepared.")

✅ Version paths prepared.


## **1.5 Reproducibility Setup**

---

In [6]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

A fixed seed is set at the start of the notebook to improve reproducibility across repeated runs of dataset construction, model initialization, and fine-tuning.

## **1.6 Load SentenceTransformer Embedde**

---

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embed_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)
print("✅ Loaded embedder:", embed_model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:01<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Loaded embedder: all-MiniLM-L6-v2


Uses the same embedder used to build the FAISS index.

# **Section 2: Shared Retrieval & Prompt Construction Utilities**

---

## **2.1 Retrieval Helper + Prompt Builder**

---

In [7]:

TOP_K = 3

def retrieve_top_k_chunks(query: str, top_k: int = TOP_K):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(q_emb, top_k)
    idxs = indices[0].tolist()

    retrieved = []
    for rank, i in enumerate(idxs):
        row = meta[i]
        retrieved.append({
            "faiss_id": i,
            "text": row.get("text", ""),
            "source": row.get("source", ""),
            "page_number": row.get("page_number", None),
            "distance": float(distances[0][rank]),
        })
    return retrieved

def build_rag_input(retrieved_chunks, question: str) -> str:
    context_blocks = []
    for c in retrieved_chunks:
        src = c["source"]
        pg = c["page_number"]
        txt = c["text"]
        context_blocks.append(f"[Source: {src} | Page: {pg}]\n{txt}")

    context = "\n\n---\n\n".join(context_blocks)

    return (
        "You are a legal assistant. Answer the question using ONLY the retrieved context.\n"
        "If the context does not contain the answer, say: \"Not found in the provided context.\"\n\n"
        f"Retrieved Context (Top-{TOP_K}):\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

def join_expected_output(exp):
    if exp is None:
        return ""
    if isinstance(exp, list):
        return "\n".join([str(x).strip() for x in exp if str(x).strip()])
    return str(exp).strip()

Defines the same retrieval + RAG formatting used in baseline, ensuring fair before/after comparison.

# **Section 3: Phase 1 - Initial Fine-Tuning (TOP_K = 3)**

---

## **3.1 Dataset Construction (TOP_K = 3)**

---

In [ ]:
from datasets import Dataset

train_rows = []
for p in prompts:
    question = p.get("question", "")
    expected = join_expected_output(p.get("expected_output"))

    # Build training input with retrieved context
    retrieved = retrieve_top_k_chunks(question, top_k=TOP_K)
    prompt_text = build_rag_input(retrieved, question)

    # Supervised target (expected answer)
    # We train the model to produce the expected output given the same RAG prompt.
    train_rows.append({
        "prompt_id": p.get("prompt_id"),
        "prompt_text": prompt_text,
        "target_text": expected
    })

ds_train = Dataset.from_list(train_rows)
print("✅ Training samples:", len(ds_train))
print(ds_train[0]["prompt_text"][:400])
print("\n--- TARGET ---\n", ds_train[0]["target_text"])

✅ Training samples: 10
You are a legal assistant. Answer the question using ONLY the retrieved context.
If the context does not contain the answer, say: "Not found in the provided context."

Retrieved Context (Top-3):
[Source: RA-10173-Data-Privacy-Act-of-2012.pdf | Page: 6]
authorizing their processing. the personal information controller must ensure implementation of personal information processing principles set out 

--- TARGET ---
 Consent of the data subject
Fulfilling contractual obligations
Compliance with legal obligations
Protection of vitally important interests
Responding to national emergency or public order
Fulfillment of statutory mandate of a public authority
Legitimate interests of controller or third parties, except when overridden by fundamental rights


Creates supervised fine-tuning samples using your existing prompts + expected outputs. This is a small dataset, but valid for demonstrating PEFT.

## **3.2 Model Initialization (4-bit QLoRA)**

---

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ⚠️ Set this to the SAME Falcon model you used in baseline
MODEL_ID = "tiiuae/falcon-7b-instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading Falcon tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=HF_CACHE_DIR, use_fast=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

gc.collect()
torch.cuda.empty_cache()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("Loading Falcon model (4-bit)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    device_map={"": 0},
    quantization_config=bnb_config,
    low_cpu_mem_usage=True,
    force_download=True
)
model.train()
print("✅ Falcon loaded for fine-tuning.")

Loading Falcon tokenizer...
Loading Falcon model (4-bit)...


config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

✅ Falcon loaded for fine-tuning.


Loads Falcon in 4-bit for QLoRA fine-tuning.

## **3.3 LoRA Configuration**

---

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 4,718,592 || all params: 7,221,908,352 || trainable%: 0.0653


Enables PEFT adapters so you train only a small number of parameters.

## **3.4 Tokenization & Loss Masking**

---

In [ ]:
MAX_LEN = 1024

def tokenize_and_mask(batch):
    prompt = batch["prompt_text"]
    target = batch["target_text"]

    full = prompt + " " + target
    tok_full = tokenizer(full, truncation=True, max_length=MAX_LEN)

    tok_prompt = tokenizer(prompt, truncation=True, max_length=MAX_LEN)
    prompt_len = len(tok_prompt["input_ids"])

    labels = tok_full["input_ids"].copy()
    # mask prompt tokens so loss is computed only on target_text
    labels[:prompt_len] = [-100] * prompt_len

    tok_full["labels"] = labels
    return tok_full

ds_train_tok = ds_train.map(tokenize_and_mask, remove_columns=ds_train.column_names)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

This trains the model to generate the expected answer, not to “memorize” the prompt text.

## **3.5 Training Run**

---

In [ ]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir=f"{PROJECT_DIR}/Models/falcon_finetune_ckpts",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds_train_tok,
    data_collator=data_collator
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,1.933827
2,2.054820
3,1.870437
4,2.217500
5,1.916004
6,1.943569
7,1.964726
8,1.657749
9,1.934116
10,1.698057


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

TrainOutput(global_step=10, training_loss=1.9190805435180665, metrics={'train_runtime': 454.5094, 'train_samples_per_second': 0.11, 'train_steps_per_second': 0.022, 'total_flos': 2071697993433600.0, 'train_loss': 1.9190805435180665, 'epoch': 5.0})

Runs PEFT fine-tuning. With only 10 samples, this is fast and mainly helps demonstrate adapter-based tuning.

## **3.6 Adapter Export**

---

In [ ]:
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("✅ Saved LoRA adapter + tokenizer to:", ADAPTER_DIR)

✅ Saved LoRA adapter + tokenizer to: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Models/falcon_finetune_adapter


Saves LoRA weights (small) so you can reload without re-training.

### **3.6.1 Artifact Version Control**

To maintain reproducibility and prevent unintended overwriting of previous experimental artifacts, Phase 1 outputs are versioned prior to proceeding with additional training runs. Existing JSON results are preserved, and the exported adapter is moved to a version-specific directory. This ensures clean separation between experimental iterations.

In [ ]:
# Preserve existing JSON results (if present)
if os.path.exists(PHASE1_JSON_CANON):
    shutil.copy(PHASE1_JSON_CANON, PHASE1_JSON_ORIGINAL)
    print("✅ Existing Phase 1 JSON preserved.")
else:
    print("⚠️ No existing Phase 1 JSON found to preserve.")

# Move current adapter to versioned folder
if os.path.exists(PHASE1_ADAPTER_CANON):
    if os.path.exists(PHASE1_ADAPTER_RERUN):
        shutil.rmtree(PHASE1_ADAPTER_RERUN)
    shutil.move(PHASE1_ADAPTER_CANON, PHASE1_ADAPTER_RERUN)
    print("✅ Phase 1 adapter moved to versioned directory.")
else:
    print("⚠️ Phase 1 adapter directory not found.")

✅ Existing Phase 1 JSON preserved.
✅ Phase 1 adapter moved to versioned directory.


## **3.7 Evaluation Model Setup**

---

In [ ]:
import gc
import torch

# Switch trained model to eval mode
model.eval()

# Free any cached memory
gc.collect()
torch.cuda.empty_cache()

ft_model = model  # use the already-loaded fine-tuned model
print("✅ Using in-memory fine-tuned model for evaluation (no reload).")

✅ Using in-memory fine-tuned model for evaluation (no reload).


Loads fine-tuned adapters on top of the base Falcon model.

## **3.8 Deterministic Generation**

---

In [ ]:
@torch.inference_mode()
def generate_answer_ft(input_text: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(input_text, return_tensors="pt").to(ft_model.device)

    out_ids = ft_model.generate(
        **inputs,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )

    decoded = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return decoded[len(input_text):].strip()

Keeps evaluation deterministic to compare fairly against baseline.

## **3.9 Evaluation & JSON Export**

---

In [ ]:
from tqdm import tqdm

FT_OUTPUT_PATH = PHASE1_JSON_RERUN

ft_results = []

for p in tqdm(prompts, desc="Running Falcon Fine-Tuned RAG prompts"):
    q = p["question"]
    retrieved = retrieve_top_k_chunks(q, top_k=TOP_K)
    input_text = build_rag_input(retrieved, q)
    answer = generate_answer_ft(input_text, max_new_tokens=256)

    ft_results.append({
        "model_id": MODEL_ID,
        "adapter_dir": ADAPTER_DIR,
        "prompt_id": p.get("prompt_id"),
        "method": p.get("method"),
        "condition": p.get("condition"),
        "dataset_file": p.get("dataset_file"),
        "section_index": p.get("section_index"),
        "question": q,
        "retrieved_chunks": retrieved,
        "answer": answer,
        "expected_output": p.get("expected_output")
    })

with open(FT_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(ft_results, f, indent=2, ensure_ascii=False)

print("✅ Saved fine-tuned outputs to:", FT_OUTPUT_PATH)

Running Falcon Fine-Tuned RAG prompts: 100%|██████████| 10/10 [02:18<00:00, 13.87s/it]

✅ Saved fine-tuned outputs to: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Results/finetuned_outputs_falcon_phase1_rerun.json


Creates the “fine-tuned Falcon outputs” file that your Original Notebook will load in Section 6/7.

## **3.10 Output Preview**

---

In [ ]:
print(f"\n✅ Loaded {len(ft_results)} fine-tuned outputs\n")

for item in ft_results:
    print("=" * 80)
    print(f"[Prompt {item['prompt_id']}] ({item['method']})")
    print("-" * 80)
    print("Q:", item["question"])
    print("\nA:", item["answer"][:400])
    print("\n")


✅ Loaded 10 fine-tuned outputs

[Prompt 1] (Zero-Shot)
--------------------------------------------------------------------------------
Q: List all the conditions under which the processing of personal information is considered lawful according to this section.

A: The processing of personal information is considered lawful according to this section when:

1. The personal information is collected from the data subject.
2. The personal information is collected for a specific purpose.
3. The personal information is collected from a reliable source.
4. The personal information is collected for a specific purpose.
5. The personal information is collected from a 


[Prompt 2] (Zero-Shot)
--------------------------------------------------------------------------------
Q: What are the penalties for unauthorized disclosure of personal information and sensitive personal information under this section?

A: The penalties for unauthorized disclosure of personal information and sensitive personal 

# **Section 4: Phase 2 - Refined Fine-Tuning (TOP_K = 5)**

---

## **4.1 Rationale for Refinement**

---

Phase 1 demonstrated a complete parameter-efficient fine-tuning pipeline but produced only marginal quantitative changes relative to the baseline model. A likely limitation is retrieval coverage. With TOP_K = 3, certain prompts may not include sufficient legal evidence in the retrieved context to fully support the expected outputs. This creates supervision mismatch, as the model is instructed to rely solely on retrieved context while being trained against complete reference answers.



To address this, Phase 2 increases retrieval coverage to TOP_K = 5 and slightly stabilizes training hyperparameters (reduced epochs and learning rate). This refinement aims to improve semantic alignment while maintaining strong contextual grounding.

## **4.2 Configuration**

---

In [8]:
TOP_K_V2 = 5

ADAPTER_DIR_V2 = f"{PROJECT_DIR}/Models/falcon_finetune_adapter_v2_topk5"
os.makedirs(ADAPTER_DIR_V2, exist_ok=True)

FT_OUTPUT_PATH_V2 = f"{RESULTS_DIR}/finetuned_outputs_falcon_v2_topk5.json"

print("✅ Phase 2 TOP_K:", TOP_K_V2)
print("✅ Phase 2 Adapter dir:", ADAPTER_DIR_V2)
print("✅ Phase 2 Output JSON:", FT_OUTPUT_PATH_V2)

✅ Phase 2 TOP_K: 5
✅ Phase 2 Adapter dir: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Models/falcon_finetune_adapter_v2_topk5
✅ Phase 2 Output JSON: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Results/finetuned_outputs_falcon_v2_topk5.json


This section defines the experimental configuration for the refined fine-tuning phase. Retrieval depth is increased to TOP_K = 5 to provide broader contextual coverage. New directories are defined for saving Phase 2 adapters and evaluation outputs to ensure that results from Phase 1 remain preserved. This separation guarantees a controlled experimental comparison between baseline, Phase 1, and Phase 2 configurations without overwriting previous artifacts.

## **4.3 Retrieval Utilities**

---

In [9]:
def retrieve_top_k_chunks_v2(query: str, top_k: int = TOP_K_V2):
    q_emb = embedder.encode([query], convert_to_numpy=True).astype(np.float32)
    distances, indices = index.search(q_emb, top_k)
    idxs = indices[0].tolist()

    retrieved = []
    for rank, i in enumerate(idxs):
        row = meta[i]
        retrieved.append({
            "faiss_id": i,
            "text": row.get("text", ""),
            "source": row.get("source", ""),
            "page_number": row.get("page_number", None),
            "distance": float(distances[0][rank]),
        })
    return retrieved


def build_rag_input_v2(retrieved_chunks, question: str, top_k: int = TOP_K_V2) -> str:
    context_blocks = []
    for c in retrieved_chunks:
        context_blocks.append(
            f"[Source: {c['source']} | Page: {c['page_number']}]\n{c['text']}"
        )

    context = "\n\n---\n\n".join(context_blocks)

    return (
        "You are a legal assistant. Answer the question using ONLY the retrieved context.\n"
        "If the context does not contain the answer, say: \"Not found in the provided context.\"\n\n"
        f"Retrieved Context (Top-{top_k}):\n{context}\n\n"
        f"Question:\n{question}\n\n"
        "Answer:"
    )

To support the updated retrieval depth, retrieval and prompt-construction utilities are adapted to dynamically incorporate TOP_K = 5 context chunks. These functions maintain the same instruction template used in Phase 1, ensuring that the only structural change between experiments is retrieval coverage. By keeping the prompt format consistent, any observed performance differences can be attributed to retrieval depth and training refinement rather than prompt redesign.

## **4.4 Dataset Construction**

---

In [14]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model_name = "all-MiniLM-L6-v2"
embedder = SentenceTransformer(embed_model_name)

train_rows_v2 = []

for p in prompts:
    question = p.get("question", "")
    expected = join_expected_output(p.get("expected_output"))

    retrieved = retrieve_top_k_chunks_v2(question, top_k=TOP_K_V2)
    prompt_text = build_rag_input_v2(retrieved, question, top_k=TOP_K_V2)

    train_rows_v2.append({
        "prompt_id": p.get("prompt_id"),
        "prompt_text": prompt_text,
        "target_text": expected
    })

ds_train_v2 = Dataset.from_list(train_rows_v2)

print("✅ Phase 2 training samples:", len(ds_train_v2))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Phase 2 training samples: 10


In this step, the supervised fine-tuning dataset is reconstructed using the expanded retrieval configuration. For each prompt, the top five semantically similar legal chunks are retrieved from the FAISS index and embedded into the RAG instruction template. The expected output remains unchanged from Phase 1, preserving evaluation consistency. This reconstruction ensures that training supervision is better aligned with available contextual evidence.

## **4.5 Tokenization & Maskingt**

---

In [15]:
from transformers import AutoTokenizer

MODEL_ID = "tiiuae/falcon-7b-instruct"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    cache_dir=HF_CACHE_DIR
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

MAX_LEN_V2 = 1024

def tokenize_and_mask_v2(batch):
    prompt = batch["prompt_text"]
    target = batch["target_text"]

    full = prompt + " " + target
    tok_full = tokenizer(full, truncation=True, max_length=MAX_LEN_V2)

    tok_prompt = tokenizer(prompt, truncation=True, max_length=MAX_LEN_V2)
    prompt_len = len(tok_prompt["input_ids"])

    labels = tok_full["input_ids"].copy()
    labels[:prompt_len] = [-100] * prompt_len
    tok_full["labels"] = labels
    return tok_full

ds_train_tok_v2 = ds_train_v2.map(
    tokenize_and_mask_v2,
    remove_columns=ds_train_v2.column_names
)

print("✅ Phase 2 tokenized dataset ready.")

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

✅ Phase 2 tokenized dataset ready.


The dataset is tokenized using the same masking strategy applied in Phase 1. Prompt tokens are masked from loss computation to ensure that gradient updates are driven exclusively by the target answer text. This approach encourages the model to learn improved answer generation behavior rather than memorizing prompt structure. Maintaining identical masking methodology ensures fairness across experimental phases.

## **4.6 Model Reinitialization**

---

In [16]:
import gc
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

gc.collect()
torch.cuda.empty_cache()

bnb_config_v2 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading base Falcon for Phase 2...")

base_model_v2 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    cache_dir=HF_CACHE_DIR,
    device_map={"": 0},
    quantization_config=bnb_config_v2,
    low_cpu_mem_usage=True
)

base_model_v2.train()
print("✅ Base model loaded for Phase 2.")

Loading base Falcon for Phase 2...


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.word_embeddings.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Base model loaded for Phase 2.


To ensure experimental integrity, Phase 2 begins from a fresh base Falcon-7B model rather than continuing from the Phase 1 adapter. This prevents compounding adapter effects and guarantees that improvements (or regressions) can be attributed solely to the refined retrieval configuration and adjusted hyperparameters. The model is again loaded in 4-bit precision to maintain computational efficiency.

## **4.7 LoRA Configuration**

---

In [17]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

base_model_v2 = prepare_model_for_kbit_training(base_model_v2)

lora_config_v2 = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model_v2 = get_peft_model(base_model_v2, lora_config_v2)
model_v2.print_trainable_parameters()

trainable params: 4,718,592 || all params: 7,221,908,352 || trainable%: 0.0653


The LoRA configuration remains consistent with Phase 1 in terms of rank, alpha, and dropout settings. By holding these parameters constant, the experiment isolates the effects of retrieval depth and learning rate adjustments. This controlled design supports a clearer interpretation of performance differences between phases.

## **4.8 Training Run**

---

In [18]:
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling

training_args_v2 = TrainingArguments(
    output_dir=f"{PROJECT_DIR}/Models/falcon_finetune_ckpts_v2_topk5",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=5,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=1,
    save_strategy="epoch",
    report_to="none"
)

data_collator_v2 = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer_v2 = Trainer(
    model=model_v2,
    args=training_args_v2,
    train_dataset=ds_train_tok_v2,
    data_collator=data_collator_v2
)

train_result_v2 = trainer_v2.train()
print(train_result_v2)

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
1,1.938330
2,2.054180
3,1.876041
4,2.217375
5,1.914222
6,1.962315
7,1.965800
8,1.656621
9,1.931000
10,1.725346


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/

TrainOutput(global_step=10, training_loss=1.9241230368614197, metrics={'train_runtime': 301.8976, 'train_samples_per_second': 0.166, 'train_steps_per_second': 0.033, 'total_flos': 2127802151731200.0, 'train_loss': 1.9241230368614197, 'epoch': 5.0})


In Phase 2, fine-tuning is executed using the reconstructed Top-5 retrieval dataset and the same QLoRA-based parameter-efficient training procedure applied in Phase 1. The training configuration is kept consistent to preserve experimental control, including the same batch size, gradient accumulation strategy, number of epochs, and learning rate. This ensures that the primary experimental difference remains the retrieval depth used to construct the contextual evidence for supervision.

## **4.9 Adapter Export**

---

In [19]:
model_v2.save_pretrained(ADAPTER_DIR_V2)
tokenizer.save_pretrained(ADAPTER_DIR_V2)

print("✅ Phase 2 adapter saved to:", ADAPTER_DIR_V2)

✅ Phase 2 adapter saved to: /content/drive/MyDrive/Mapua /3rd Year Term 2/CSS181-3/NLP_Project/Models/falcon_finetune_adapter_v2_topk5


Upon completion of training, the refined LoRA adapter is saved to a distinct directory. This ensures reproducibility and enables independent evaluation of Phase 2 results without interfering with prior experiments. Maintaining separate adapter artifacts supports clean comparative analysis.

## **4.10 Evaluation & JSON Export**

---

In [20]:
from tqdm import tqdm

model_v2.eval()

@torch.inference_mode()
def generate_answer_v2(input_text: str, max_new_tokens: int = 256) -> str:
    inputs = tokenizer(input_text, return_tensors="pt").to(model_v2.device)
    out_ids = model_v2.generate(
        **inputs,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.eos_token_id
    )
    decoded = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    return decoded[len(input_text):].strip()

ft_results_v2 = []

for p in tqdm(prompts, desc="Running Falcon Phase 2 (TopK=5)"):
    q = p["question"]
    retrieved = retrieve_top_k_chunks_v2(q, top_k=TOP_K_V2)
    input_text = build_rag_input_v2(retrieved, q, top_k=TOP_K_V2)
    answer = generate_answer_v2(input_text)

    ft_results_v2.append({
        "model_id": MODEL_ID,
        "adapter_dir": ADAPTER_DIR_V2,
        "prompt_id": p.get("prompt_id"),
        "method": p.get("method"),
        "condition": p.get("condition"),
        "dataset_file": p.get("dataset_file"),
        "section_index": p.get("section_index"),
        "question": q,
        "retrieved_chunks": retrieved,
        "answer": answer,
        "expected_output": p.get("expected_output"),
        "phase": "Phase 2 (TopK=5)"
    })

import json
with open(FT_OUTPUT_PATH_V2, "w", encoding="utf-8") as f:
    json.dump(ft_results_v2, f, indent=2, ensure_ascii=False)

print("✅ Phase 2 outputs saved.")

Running Falcon Phase 2 (TopK=5):   0%|          | 0/10 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
This is a friendly reminder - the current text generation call has exceeded the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degradation, or nothing at all.
Running Falcon Phase 2 (TopK=5): 100%|██████████| 10/10 [04:26<00:00, 26.60s/it]


✅ Phase 2 outputs saved.


The fine-tuned Phase 2 model is evaluated using deterministic generation settings identical to those used in Phase 1 and the baseline evaluation. Responses are generated for the same set of prompts and saved to a new JSON file. This guarantees methodological consistency and enables direct quantitative comparison across all three experimental configurations.

## **4.11 Output Preview**

---

In [21]:
print(f"\n✅ Loaded {len(ft_results_v2)} Phase 2 fine-tuned outputs\n")

for item in ft_results_v2:
    print("=" * 80)
    print(f"[Prompt {item['prompt_id']}] ({item['method']})")
    print("-" * 80)
    print("Q:", item["question"])
    print("\nA:", item["answer"][:400])
    print("\n")


✅ Loaded 10 Phase 2 fine-tuned outputs

[Prompt 1] (Zero-Shot)
--------------------------------------------------------------------------------
Q: List all the conditions under which the processing of personal information is considered lawful according to this section.

A: The processing of personal information is considered lawful according to this section if:

1. The data subject has given his or her consent.
2. The processing is necessary and is related to the fulfillment of a contract with the data subject or in order to take steps at the request of the data subject prior to entering into a contract.
3. The processing is necessary for compliance with a legal obl


[Prompt 2] (Zero-Shot)
--------------------------------------------------------------------------------
Q: What are the penalties for unauthorized disclosure of personal information and sensitive personal information under this section?

A: The penalties for unauthorized disclosure of personal information and sensitive p

This section provides a preview of the Phase 2 generated responses after evaluation and JSON export. Displaying the outputs directly in the notebook allows quick verification that the refined Top-5 configuration produced valid responses for each prompt before formal comparison in the succeeding section.

# **Section 5: Phase 1 vs Phase 2 Result Comparison**

---

### **5.1 Load Saved Experimental Outputs**

---

In [24]:
PHASE1_RESULTS_PATH = f"{PROJECT_DIR}/Results/finetuned_outputs_falcon_phase1_original.json"
PHASE2_RESULTS_PATH = f"{PROJECT_DIR}/Results/finetuned_outputs_falcon_v2_topk5.json"

with open(PHASE1_RESULTS_PATH, "r", encoding="utf-8") as f:
    phase1_results = json.load(f)

with open(PHASE2_RESULTS_PATH, "r", encoding="utf-8") as f:
    phase2_results = json.load(f)

print("✅ Phase 1 outputs loaded:", len(phase1_results))
print("✅ Phase 2 outputs loaded:", len(phase2_results))

✅ Phase 1 outputs loaded: 10
✅ Phase 2 outputs loaded: 10


This subsection reloads the saved JSON outputs from Phase 1 and Phase 2 so that both experimental configurations can be reviewed within the same notebook session. Using the exported files also helps preserve reproducibility by ensuring that the displayed outputs correspond to the saved experimental artifacts.

### **5.2 Side-by-Side Comparison**

---

In [27]:
phase1_by_id = {item["prompt_id"]: item for item in phase1_results}
phase2_by_id = {item["prompt_id"]: item for item in phase2_results}

all_prompt_ids = sorted(set(phase1_by_id.keys()) | set(phase2_by_id.keys()))

for pid in all_prompt_ids:

    p1 = phase1_by_id.get(pid)
    p2 = phase2_by_id.get(pid)

    question = None
    expected = None

    if p1:
        question = p1.get("question")
        expected = p1.get("expected_output")
    elif p2:
        question = p2.get("question")
        expected = p2.get("expected_output")

    print("=" * 100)
    print(f"Prompt ID: {pid}")

    print("\nQuestion:")
    print(question)

    print("\nExpected:")
    print(expected)

    if p1:
        print("\n[Phase 1]")
        print(p1["answer"][:600])

    if p2:
        print("\n[Phase 2]")
        print(p2["answer"][:600])

    print("\n")

Prompt ID: 1

Question:
List all the conditions under which the processing of personal information is considered lawful according to this section.

Expected:
['Consent of the data subject', 'Fulfilling contractual obligations', 'Compliance with legal obligations', 'Protection of vitally important interests', 'Responding to national emergency or public order', 'Fulfillment of statutory mandate of a public authority', 'Legitimate interests of controller or third parties, except when overridden by fundamental rights']

[Phase 1]
The processing of personal information is considered lawful according to this section when:

1. The personal information is collected from the data subject.
2. The personal information is collected for a specific purpose.
3. The personal information is collected from a public authority or a government entity.
4. The personal information is collected from a public body or a government entity.
5. The personal information is collected from a public body or a governme